In [ ]:
# Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns


In [ ]:
# Load the all dataset

In [ ]:
gold_loans = pd.read_csv('/data/gold_loans.csv')
repayments = pd.read_csv('/data/repayments.csv')
gold_rates = pd.read_csv('/data/gold_rates.csv')
branches = pd.read_csv('/data/branches.csv')
customers = pd.read_csv('/data/customers.csv')

In [ ]:
# Copy all the original dataset

In [ ]:
df_gl = gold_loans.copy()
df_rp = repayments.copy()
df_gr = gold_rates.copy()
df_br = branches.copy()
df_cs = customers.copy()

In [ ]:
# Solve the Questions

In [ ]:
# 1. Size of the loan extract : How many rows are in gold_loans exactly as received (before any cleaning)?

print('Total Number of Rows in Gold Loans:', df_gl.shape[0])

In [ ]:
# 2. Size of the repayments table : How many rows are in repayments?

print('Total Number of Rows in Repayments:', df_rp.shape[0])

In [ ]:
# 3. Branch network : How many branches does Suvarna Finance have?

print('Total Number of Branches in Suvarna Finance:', df_br['branch_id'].nunique())

In [ ]:
# 4. States covered : In how many different states does the company have branches?

print('Total Number of Different States Covered by Suvarna Finance:', df_br['state'].nunique())

In [ ]:
# 5. Disbursal period : Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal.

answer = df_gl['disbursal_date'].agg({
    'first_disbursal' : 'min',
    'last_disbursal' : 'max'
}).to_frame(name='disbursal_period').T

answer

In [ ]:
# 6. Unique loans : How many distinct loan_id values are there?

print('Total Number of Unique Loans:', df_gl['loan_id'].nunique())

In [ ]:
# 7. Duplicate rows : How many exact duplicate rows must be removed from gold_loans?

print('Total Number of Exact Duplicate Rows in Gold Loans:', df_gl.duplicated().sum())

In [ ]:
# Drop duplicated rows

df_gl.drop_duplicates(inplace=True)

# After Removal of duplicates again check duplicated records
print('Total Duplicate Rows in Gold Loans After Removal:', df_gl.duplicated().sum())

In [ ]:
# 8. Inconsistent status values : In how many different ways is status written in the raw data (count distinct raw values, letter case included)?

print('Total Number of Distinct Status Values in Raw Data:', df_gl['status'].nunique())

In [ ]:
# Check Inconsistent Status Values

print('Unique Status Values Before Standardization:', df_gl['status'].unique())

In [ ]:
# Standardize Inconsistent Status Values

df_gl['status'] = df_gl['status'].str.strip().str.lower()

print('Unique Status Values After Standardization:', df_gl['status'].unique())

In [ ]:
# 9. Loans by status : After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status.

answer = df_gl['status'].value_counts().rename('loans')
answer

In [ ]:
# 10. Purity formats : How many different raw values does the purity column contain?

print('Total Number of Different Raw Values in Purity Column:', df_gl['purity'].nunique())

In [ ]:
# Check Inconsistent Purity Values

print('Unique Purity Values Before Standardization:', df_gl['purity'].unique())

In [ ]:
# Standardize Inconsistent Purity Values into one scale (karat)

df_gl['purity'] = df_gl['purity'].replace(r'\s|K|T', '', regex=True)

print('Unique Status Values After Standardization:', df_gl['purity'].unique())


In [ ]:
for index, i in df_gl['purity'].items():
    if len(str(i)) == 3:
        hallmark = int(i)
        karat = round((hallmark / 1000) * 24)

        df_gl.loc[index, 'purity'] = str(karat)

In [ ]:
print('Unique Status Values After Standardization:', df_gl['purity'].unique())


In [ ]:
#After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24),
# return karat and loans for each karat

karat={'916':22,'833':20,'750':18,'999':24}

def to_karat(p):
    num=''.join(i for i in str(p) if i.isdigit())
    return karat.get(num,int(num))

clean_loans['purity']=clean_loans['purity'].apply(to_karat)

In [ ]:
#After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24),
# return karat and loans for each karat

loans['purity'].apply(
    lambda p:karat.get(''.join(filter(str.isdigit,str(p))),
    int(''.join(filter(str.isdigit,str(p)))))
)

In [ ]:
(loans['purity']
  .str.replace('KT','')
  .str.replace('K','')
  .str.strip()
  .replace({'916':22,'833':20,'750':18,'999':24})
  .astype(int))

In [ ]:
# 11. Loans by karat : After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24), return karat and loans for each karat.

answer = df_gl['purity'].value_counts().rename_axis('karat').rename('loans')
answer

In [ ]:
# 12. Missing gold weight : After removing duplicates, how many loans have no gold weight recorded?

answer = df_gl['gold_weight_gm'].isna().sum()
answer

In [ ]:
# 13. Invalid principal : After removing duplicates, how many loans have a principal of 0 or less?

answer = len(df_gl[df_gl['principal'] <= 0])
answer

In [ ]:
# 14. Active loans : How many active loans are there after removing duplicates (status 'active' in any letter case)?

answer = len(df_gl[df_gl['status'] == 'active'])
answer

In [ ]:
# 15. Loans that cannot be assessed : How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?

df_gl[(df_gl['status'] == 'active') and (df_gl['principal'] <= 0)]